In [7]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
from algorithms.svd_manual import compute_svd_manual
from utils.helpers import print_matrix, matrix_norm
from utils.generate_matrix import generate_matrix
from benchmark import run_power_method_experiment, plot_results

NORMS = [1, 2, 3, 4, np.inf]
SEED = 17

## Punkt 1 - Metoda potęgowa dla macierzy 3 x 3

In [8]:
A = generate_matrix()
print(np.array2string(A, separator=', '))
results = run_power_method_experiment(A)
plot_results(results)

[[10.18401738, -3.74854635,  1.39947976],
 [-3.74854635, 10.14797067, -0.06626128],
 [ 1.39947976, -0.06626128, 11.46948909]]


## Punkt 2 – Ręczne SVD przez eigendecomposition AA^T

In [2]:
rng = np.random.default_rng(SEED)
A = rng.uniform(-5.0, 5.0, size=(3, 3))
B = A @ A.T

print_matrix(A, "Macierz A (seed=17)")
print_matrix(B, "B = AA^T")


Macierz A (seed=17):
[[ 3.450748 -3.390269  0.577445]
 [-1.319201 -2.85058  -1.14176 ]
 [-0.718355  1.113431  2.363859]]

B = AA^T:
[[23.735029  4.452702 -4.888694]
 [ 4.452702 11.169714 -4.925229]
 [-4.888694 -4.925229  7.343591]]


In [3]:
U, D, V = compute_svd_manual(A)

print_matrix(U, "U (lewe wektory singularne – kolumny)")
print_matrix(D, "D (diagonalna – wartości singularne)")
print_matrix(V, "V (prawe wektory singularne – kolumny)")

A_manual = U @ D @ V.T
print_matrix(A_manual, "Rekonstrukcja  U · D · V^T")
print_matrix(A, "Oryginalna A")


U (lewe wektory singularne – kolumny):
[[-0.888829 -0.448458  0.094169]
 [-0.342135  0.786172  0.514661]
 [ 0.304837 -0.425228  0.852206]]

D (diagonalna – wartości singularne):
[[5.208229 0.       0.      ]
 [0.       3.360614 0.      ]
 [0.       0.       1.956773]]

V (prawe wektory singularne – kolumny):
[[-0.544285 -0.6782   -0.493759]
 [ 0.831006 -0.355327 -0.427985]
 [ 0.114814 -0.643263  0.756988]]

Rekonstrukcja  U · D · V^T:
[[ 3.450748 -3.390269  0.577445]
 [-1.319201 -2.85058  -1.14176 ]
 [-0.718355  1.113431  2.363859]]

Oryginalna A:
[[ 3.450748 -3.390269  0.577445]
 [-1.319201 -2.85058  -1.14176 ]
 [-0.718355  1.113431  2.363859]]


## Punkt 4 – Porównanie z numpy.linalg.svd

In [4]:
U_np, s_np, VT_np = np.linalg.svd(A)
S_np = np.zeros((3, 3))
np.fill_diagonal(S_np, s_np)
A_numpy = U_np @ S_np @ VT_np

print_matrix(U_np, "U (numpy)")
print_matrix(np.diag(s_np), "S (numpy – wartości singularne na diagonali)")
print_matrix(VT_np.T, "V  (numpy)")


U (numpy):
[[-0.888829  0.448458  0.094169]
 [-0.342135 -0.786172  0.514661]
 [ 0.304837  0.425228  0.852206]]

S (numpy – wartości singularne na diagonali):
[[5.208229 0.       0.      ]
 [0.       3.360614 0.      ]
 [0.       0.       1.956773]]

V  (numpy):
[[-0.544285  0.6782   -0.493759]
 [ 0.831006  0.355327 -0.427985]
 [ 0.114814  0.643263  0.756988]]


In [5]:
fmt_p = lambda p: "∞" if p == np.inf else str(p)

print(f"\n{'Norma':<8} {'‖UDVᵀ−A‖ ręczne':>22} {'numpy':>20}")
print("  " + "-" * 48)
for p in NORMS:
    e_man = matrix_norm(A_manual - A, p)
    e_np = matrix_norm(A_numpy - A, p)
    print(f"  p={fmt_p(p):<6} {e_man:>22.8e} {e_np:>20.8e}")


Norma           ‖UDVᵀ−A‖ ręczne                numpy
  ------------------------------------------------
  p=1              4.44089210e-15       1.33226763e-14
  p=2              1.94209711e-15       5.00463123e-15
  p=3              1.57591301e-15       3.83102716e-15
  p=4              1.45251198e-15       3.44994887e-15
  p=∞              1.33226763e-15       3.10862447e-15
